# 01 — Physical samples and a detector-scale uncertainty

This series starts again from the five-dimensional Gaussian-mixture example. We keep its signal, background, latent variables, and detector response, and remove the preselection entirely. This notebook creates the independent samples used by the later notebooks and shows what the scale variation changes.

The nominal expected yields are $S=100$ and $B=1\,000$, so $S/B=1/10$ at signal strength $\mu=1$. These are expected **experimental counts**, not the number of Monte Carlo events used for training. We can generate millions of training events while a typical experiment contains about $1\,100$ events.

Notebook 02 learns the reference density and process ratios; notebook 03 introduces the controlled signal-model error; notebook 04 learns the response and profiling optimizers; notebook 05 calibrates the resulting statistic distributions.

## Setup and shared run directory

Run this cell first. Use the same `RUN_NAME` in all five notebooks. Colab can keep the run on Drive; locally, open Jupyter in this directory. Use a fresh kernel when switching from the older exercises. Training and samples remain in the run directory.


In [ ]:
# Run this cell first in each notebook. All five notebooks share RUN_NAME.
import os
import sys
import subprocess
from pathlib import Path

TAG = "sb10_exppoly_10m_slowlr_v3_20261001"  # New tag: rebuild all five stages.
RUN_NAME = TAG
USE_DRIVE = True
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = Path("/content/nsbi_calibration_toolkit")
    BRANCH = "ml4hep_school_tutorial"
    if not (REPO / ".git").exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH,
            "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git", str(REPO),
        ], check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
        subprocess.run([
            "git", "-C", str(REPO), "sparse-checkout", "set", "src",
            "workshops/ml4hep_tifr_colab/calibration",
        ], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
    SOURCE = REPO / "workshops/ml4hep_tifr_colab/calibration"
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "-r",
        str(SOURCE / "requirements.txt"),
    ], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        RUN_BASE = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab/calibration_runs")
    else:
        RUN_BASE = Path("/content/calibration_runs")
else:
    # Open the notebook with calibration/ as its working directory.
    SOURCE = Path.cwd().resolve()
    REPO = SOURCE.parents[2]
    RUN_BASE = SOURCE / "runs"

sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(SOURCE))
os.chdir(SOURCE)
RUN = Path(os.environ.get("CALIBRATION_RUN", RUN_BASE / RUN_NAME)).resolve()
RUN.mkdir(parents=True, exist_ok=True)
print("Source:", SOURCE)
print("TAG:", TAG)
print("Run:", RUN)
if "CALIBRATION_RUN" in os.environ:
    print("CALIBRATION_RUN overrides the tagged path; use a fresh directory for this model.")
print("Use a fresh kernel when switching between the old exercises and calibration.")


In [ ]:
import hashlib
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LogNorm

from model import DEFAULT_CONFIG, save_config
from sampling import FEATURES, read_sample, sample_experiment, sample_process

PLOT_DIR = RUN / "plots" / "01_samples"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Define the experiment and training budgets

There are five observed variables $x_1,\ldots,x_5$. A latent Gaussian-mixture event $z$ is measured through

$$x_j=s_jz_j+\epsilon_j,\qquad \epsilon_j\sim\mathcal N(0,\sigma_j^2).$$

The up/down anchors multiply the response **mean** by $1.1$ or $0.9$, keeping the resolution $\sigma_j$ fixed. They do not rescale the complete observed vector, because that would also rescale the resolution. The copied generator retains the latent event and its resolution residual when making these paired anchors.

All features have their complete real-valued support. A plot may display a finite window, but no event is rejected by a feature cut.

The budgets below use 500,000 events per process for the balanced reference-flow training, 10,000,000 events per class for nominal ratios, and 250,000 independent evaluation events per process. Systematic ratios start from 1,000,000 events per class. The 50:50 reference mixture used to train the flow is a computational proposal, independent of the experimental $S/B$.

In [ ]:
config = dict(DEFAULT_CONFIG)
config["samples"] = {
    "flow_train_per_process": 500_000,
    "nominal_ratio_per_class": 10_000_000,
    "evaluation_per_process": 250_000,
    "variation_ratio_per_class": 1_000_000,
}
save_config(config, RUN)

sizes = config["samples"]
rows_per_process = (
    sizes["flow_train_per_process"]
    + sizes["nominal_ratio_per_class"]
    + sizes["evaluation_per_process"]
)
display(pd.DataFrame({
    "sample block": ["Reference flow", "Nominal ratios", "Independent evaluation"],
    "events per process": [sizes["flow_train_per_process"],
                           sizes["nominal_ratio_per_class"],
                           sizes["evaluation_per_process"]],
}))
print(f"Signal yield: {config['signal_yield']:g}; background yield: {config['background_yield']:g}")
print(f"S/B at mu=1: {config['signal_yield']/config['background_yield']:.4f}")
print(f"Rows per process and anchor: {rows_per_process:,}")

## 2. Generate the nominal and varied samples

Each process has three parquet files: nominal, scale down, and scale up. Their row indices refer to the same latent event and resolution residual. We reserve whole row blocks before any model training:

1. The first block trains the reference flow.
2. The next block supplies process-ratio training samples.
3. The last block is held out from all flow and ratio training.

An evaluation event and both of its variations therefore remain in the evaluation block. Within the ratio block, notebook 02 uses the smaller systematic budget when training the deformation ratios. The additional anchor rows are stored so every partition follows the same simple rule.

Generation is streamed in batches of 100,000 events. The full default sample set contains 34.5 million rows across six files, so allow a few GB of disk space. A manifest records the settings and generator source; rerunning this cell reuses the completed matching files. If you change these settings, use a new `RUN` directory for the full series.

In [ ]:
sample_dir = RUN / "samples"
sample_dir.mkdir(parents=True, exist_ok=True)
generator = Path("generate_distributions.py").resolve()
distributions = Path("utils_distributions.py").resolve()
recipe = {
    "rows_per_process": rows_per_process,
    "seed": config["seed"],
    "signal_yield": config["signal_yield"],
    "background_yield": config["background_yield"],
    "generator_sha256": hashlib.sha256(generator.read_bytes()).hexdigest(),
    "distributions_sha256": hashlib.sha256(distributions.read_bytes()).hexdigest(),
    "samples": sizes,
}
manifest = sample_dir / "manifest.json"
expected_files = [
    sample_dir / f"{process}{suffix}.parquet"
    for process in ["signal", "background"]
    for suffix in ["", "_scale_down", "_scale_up"]
]
complete = (
    manifest.exists()
    and json.loads(manifest.read_text()) == recipe
    and all(path.exists() for path in expected_files)
)
if complete:
    print("Reusing the matching sample set.")
else:
    subprocess.run([
        sys.executable, str(generator),
        "--n_sig", str(rows_per_process),
        "--n_bkg", str(rows_per_process),
        "--signal-yield", str(config["signal_yield"]),
        "--background-yield", str(config["background_yield"]),
        "--seed", str(config["seed"]),
        "--output-dir", str(sample_dir.resolve()),
        "--generation-batch-size", "100000",
        "--systematics-batch-size", "100000",
        "--with-systematics", "--skip-data", "--skip-plots",
    ], check=True)
    manifest.write_text(json.dumps(recipe, indent=2))
print(sample_dir.resolve())

## 3. Look at the five observed variables

The upper row compares unit-normalized shapes. The lower row weights the same independent events to the expected yields $S$ and $B$. This separates shape discrimination from the small signal fraction.

For readable axes, the displayed bins span the central 99% of the combined diagnostic sample. Underflow and overflow events still exist in the model and remain in the data files. The histograms use weights normalized over the **complete** sample, not only the displayed range.

In [ ]:
plot_n = min(50_000, sizes["evaluation_per_process"])
nominal = {
    process: read_sample(RUN, process, partition="eval", n=plot_n)
    for process in ["signal", "background"]
}
colors = {"signal": "tab:red", "background": "tab:blue"}
bins_by_feature = [
    np.linspace(*np.quantile(np.concatenate([nominal[p][:, j] for p in nominal]),
                            [0.005, 0.995]), 51)
    for j in range(len(FEATURES))
]
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for j, feature in enumerate(FEATURES):
    bins = bins_by_feature[j]
    width = bins[1] - bins[0]
    for process, values in nominal.items():
        unit_weight = np.full(len(values), 1.0 / len(values))
        axes[0, j].hist(values[:, j], bins=bins, weights=unit_weight / width,
                        histtype="step", color=colors[process], label=process)
        expected_yield = config[f"{process}_yield"]
        axes[1, j].hist(values[:, j], bins=bins, weights=unit_weight * expected_yield,
                        histtype="step", color=colors[process], label=process)
    axes[0, j].set_title(feature)
    axes[1, j].set_xlabel(feature)
    axes[1, j].set_yscale("log")
axes[0, 0].set_ylabel("Probability density")
axes[1, 0].set_ylabel("Expected events per bin")
axes[0, 0].legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / "nominal_marginals.png", dpi=150)
plt.show()

## 4. Check correlations and joint structure

The marginals do not specify the full experiment. The components have different means and correlated covariance matrices, and the resulting mixture need not be Gaussian. These plots make the joint structure visible before we ask a density estimator to learn it.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for j, process in enumerate(["signal", "background"]):
    values = nominal[process]
    image = axes[0, j].imshow(np.corrcoef(values, rowvar=False), vmin=-1, vmax=1,
                              cmap="coolwarm")
    axes[0, j].set_xticks(range(5), FEATURES)
    axes[0, j].set_yticks(range(5), FEATURES)
    axes[0, j].set_title(f"{process}: correlations")
    fig.colorbar(image, ax=axes[0, j])
    axes[1, j].hist2d(values[:, 0], values[:, 1],
                      bins=[bins_by_feature[0], bins_by_feature[1]],
                      norm=LogNorm(), cmap="viridis")
    axes[1, j].set(xlabel="x1", ylabel="x2", title=f"{process}: joint density")
fig.tight_layout()
fig.savefig(PLOT_DIR / "joint_structure.png", dpi=150)
plt.show()

## 5. Define the scale uncertainty

The physical simulator changes the detector-response mean continuously:

$$x_j=(1+0.1\alpha)s_jz_j+\epsilon_j,\qquad
\epsilon_j\sim\mathcal N(0,\sigma_j^2),\qquad -1\leq\alpha\leq1.$$

Every process remains normalized and its expected yield stays fixed. The nominal and $\pm1$ samples are the same three detector-scale anchors used previously; intermediate values now correspond to a continuous detector-scale change.

Notebook 02 models this nuisance using the paper's polynomial interpolation, with exponential extrapolation outside the anchors (the fit remains restricted to $[-1,1]$). It normalizes each complete interpolated process density at every nuisance value. A sparse positivity correction preserves the three anchors and smooth joins if the polynomial becomes negative in a tail. The notebook reports its size.

The simulator and fitted interpolation agree at the physical anchor definitions, up to learned-density error; intermediate-point closure is a separate check of the interpolation approximation. Reference-model toys follow the fitted normalized interpolation. Solid curves below are nominal and dashed curves are the two anchors.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for row, process in enumerate(["signal", "background"]):
    for anchor, color, style in [("down", "tab:blue", "--"),
                                  ("nominal", "black", "-"),
                                  ("up", "tab:orange", "--")]:
        values = (nominal[process] if anchor == "nominal" else
                  read_sample(RUN, process, partition="eval", anchor=anchor, n=plot_n))
        for j, feature in enumerate(FEATURES):
            bins = bins_by_feature[j]
            weights = np.full(len(values), 1.0 / (len(values) * (bins[1] - bins[0])))
            axes[row, j].hist(values[:, j], bins=bins, weights=weights, histtype="step",
                              color=color, linestyle=style, label=anchor)
            axes[row, j].set_xlabel(feature)
    axes[row, 0].set_ylabel(f"{process}\nProbability density")
    axes[row, 0].legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / "scale_anchors.png", dpi=150)
plt.show()

## 6. What is one pseudo-experiment?

At generating parameters $(\mu,\alpha)$, draw independent counts

$$N_S\sim\operatorname{Pois}(\mu S),\qquad N_B\sim\operatorname{Pois}(B),$$

then draw fresh unbinned signal and background events. The observed experiment is the unordered collection $D=\{x_i\}_{i=1}^N$ together with an auxiliary observable $y$.

As requested, our toy ensemble draws $y\sim\mathcal U(-2,2)$. In contrast, the analysis likelihood contains the Gaussian constraint

$$L_{\rm aux}(\alpha;y)\propto\exp[-(y-\alpha)^2/2].$$

These two statements serve different purposes: the first specifies the ensemble used to train and validate; the second specifies the objective being fitted. Coverage measured later is coverage for this **uniform-auxiliary ensemble**, at the stated generating nuisance value. It is not automatically coverage for repeated Gaussian auxiliary measurements or for every possible nuisance value.

The function below draws directly from the analytic physical simulator, without resampling a finite Monte Carlo bank. We will use these fresh experiments to validate the learned inference machinery.

In [ ]:
rng = np.random.default_rng(config["seed"] + 900)
toy = sample_experiment(mu=1.0, alpha=0.0, config=config, rng=rng)
print(f"Fresh experiment: N={len(toy['x']):,}, auxiliary={toy['auxiliary']:.3f}")
print("Observed event array:", toy["x"].shape)

# A visual check of the specified auxiliary ensemble and Poisson count model.
auxiliary_draws = rng.uniform(-2.0, 2.0, size=5000)
count_draws = rng.poisson(config["signal_yield"] + config["background_yield"], size=5000)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(auxiliary_draws, bins=40, density=True, histtype="step")
axes[0].axhline(0.25, color="black", linestyle="--", label="Uniform density")
axes[0].set(xlabel="Auxiliary observable y", ylabel="Density")
axes[0].legend()
axes[1].hist(count_draws, bins=40, histtype="step")
axes[1].set(xlabel="Total event count N at mu=1", ylabel="Experiments")
fig.tight_layout()
fig.savefig(PLOT_DIR / "experiment_ensemble.png", dpi=150)
plt.show()

The files in `RUN/samples/` and the shared `RUN/config.json` are the inputs to notebook 02. The physical sampling functions in `sampling.py` also remain available to all later notebooks, so validation toys can be freshly generated instead of reused from training.